#### console (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:! console_color = $'System.ConsoleColor'

:= reset_color () : () =
    run_target \|
        | Fsharp => \ () => $'System.Console.ResetColor ()'
        | _ => \ () => ()

:= set_foreground_color (color : console_color) : () =
    run_target \|
        | Fsharp => \ () => $'System.Console.ForegroundColor <- !color '
        | _ => \ () => ()

:= write_line_c \* t. (obj : t) : () =
    ~>
        ?: t ?|
        | string => $'printf("%s\\n", (!obj)->ptr)' : ()
        | i32 => $'printf("%d\\n", !obj)' : ()
        | i64 => $'printf("%lld\\n", (long long)(!obj))' : ()
        | u32 => $'printf("%u\\n", !obj)' : ()
        | u64 => $'printf("%llu\\n", (unsigned long long)(!obj))' : ()
        | _ => $'backend_switch_has_no_C_arm_in_lib_spiral' : ()

:= write_line_gleam \* t. (obj : t) : () =
    global "import gleam/io"
    ? (~> ?: t ?| string => !+ | _ => !-) ?> $'io.println(!obj)' : ()
    |? (~> ?: t ?| () => !+ | _ => !-) ?> $'io.println("")' : ()
    |:
        global "import gleam/int"
        $'io.println(int.to_string(!obj))' : ()

:= write_wasm \* t. (obj : t) : () =
    ~>
        ?: t ?|
        | string => $'(call $print_str !obj)' : ()
        | i32 => $'(call $print_i64 (i64.extend_i32_s !obj))' : ()
        | i64 => $'(call $print_i64 !obj)' : ()
        | u32 => $'(call $print_u64 (i64.extend_i32_u !obj))' : ()
        | u64 => $'(call $print_u64 !obj)' : ()
        | () => $'' : ()
        | _ => $'backend_switch_has_no_Wasm_arm_in_lib_spiral' : ()

:= write_line_bend \* t. (x : t) : () =
    ~>
        ?: t ?|
        | string => !!!!Printf("%s\n", x)
        | i32 => !!!!Printf("%d\n", x)
        | _ => $'backend_switch_has_no_Bend_arm_in_lib_spiral' : ()

:= write_line_lua \* t. (obj : t) : () =
    ~>
        ?: t ?|
        | i64 => $'print(spiral_w64.tostring(!obj, true))' : ()
        | u64 => $'print(spiral_w64.tostring(!obj, false))' : ()
        | _ => $'print(!obj)' : ()

:= write_line \* t. (obj : t) : () =
    backend_switch {
        Gleam = \ () => write_line_gleam obj
        Lua = \ () => write_line_lua obj
        Fsharp = \ () =>
            \ () => obj |> $'System.Console.WriteLine'
            |> exec_unit
        Python = \ () => $'print(!obj)' : ()
        Rust = \ () => $'println\!("{}", !obj)' : ()
        TypeScript = \ () => $'process.stdout.write(String(!obj) + "\\n")' : ()
        Zig = \ () =>
            $'spiralPrintAny(!obj)' : ()
            $'spiralPrint(&[_]u8{10})' : ()
        Lean = \ () => $'IO.println (toString !obj)' : ()
        C = \ () => write_line_c obj
        Pascal = \ () => $'Writeln(!obj)' : ()
        Kotlin = fun () => $'run { System.out.write((!obj.toString() + "\\n").toByteArray(Charsets.UTF_8)); \
            System.out.flush() }' : ()
        Swift = \ () => $'print(!obj)' : ()
        Bend = \ () => write_line_bend obj
        Wasm = \ () =>
            write_wasm obj
            $'(call $print_newline)' : ()
    }

:= write_error_line \* t. (obj : t) : () =
    backend_switch {
        Gleam = \ () =>
            global "import gleam/io"
            $'io.println_error(!obj)' : ()
        Lua = \ () => $'io.stderr:write(tostring(!obj) .. "\\n")' : ()
        Fsharp = \ () =>
            \ () => obj |> $'System.Console.Error.WriteLine'
            |> exec_unit
        Python = \ () =>
            global "import sys"
            $'print(!obj, file=sys.stderr)' : ()
        Rust = \ () => $'eprintln\!("{}", !obj)' : ()
        TypeScript = \ () => $'process.stderr.write(String(!obj) + "\\n")' : ()
        Lean = \ () => $'IO.eprintln (toString !obj)' : ()
        Kotlin = fun () => $'run { System.err.write((!obj.toString() + "\\n").toByteArray(Charsets.UTF_8)); \
            System.err.flush() }' : ()
    }

#### write_error_line_writes_without_failing (test)

In [1]:
///- --test --symbol
///> rust

write_error_line "console.write_error_line probe"

console.write_error_line probe

0

In [ ]:
:! stdin_handle = $'std::io::Stdin'

:! stdin_error = $'std::io::Error'

:= read_to_end () : string =
    backend_switch {
        Rust = \ () =>
            := handle = $'std::io::stdin()' : stdin_handle
            := read = $'std::io::read_to_string(!handle)' : resultm::result' sm'::std_string stdin_error
            read |> resultm::unwrap_or' (sm'::to_std_string "") |> sm'::from_std_string
        Fsharp = \ () => $'System.Console.In.ReadToEnd ()' : string
        Python = \ () =>
            global "import sys"
            $'sys.stdin.read()' : string
    }

#### read_to_end_types_without_reading (test)

In [2]:
///- --test --symbol
///> rust

:= far = sm'::length "x" > 1i32
? far ?> read_to_end () |> ignore |: ()

0

In [ ]:
:= write_c \* t. (x : t) : () =
    ~>
        ?: t ?|
        | string => $'printf("%s", (!x)->ptr)' : ()
        | i32 => $'printf("%d", !x)' : ()
        | i64 => $'printf("%lld", (long long)(!x))' : ()
        | u32 => $'printf("%u", !x)' : ()
        | u64 => $'printf("%llu", (unsigned long long)(!x))' : ()
        | _ => $'backend_switch_has_no_C_arm_in_lib_spiral' : ()

:= write_gleam \* t. (x : t) : () =
    global "import gleam/io"
    ? (~> ?: t ?| string => !+ | _ => !-) ?> $'io.print(!x)' : ()
    |:
        global "import gleam/int"
        $'io.print(int.to_string(!x))' : ()

:= write_lua \* t. (x : t) : () =
    ~>
        ?: t ?|
        | i64 => $'io.write(spiral_w64.tostring(!x, true))' : ()
        | u64 => $'io.write(spiral_w64.tostring(!x, false))' : ()
        | _ => $'io.write(!x)' : ()

:= write \* t. (x : t) : () =
    backend_switch {
        Gleam = \ () => write_gleam x
        Lua = \ () => write_lua x
        Fsharp = \ () => $'!x |> System.Console.Write' : ()
        Python = \ () => $'print(!x, end="")' : ()
        Rust = \ () =>
            $'print\!("{}", !x)' : ()
            $'std::io::Write::flush(&mut std::io::stdout()).unwrap()' : ()
        TypeScript = \ () => $'process.stdout.write(String(!x))' : ()
        Zig = \ () => $'spiralPrintAny(!x)' : ()
        Lean = \ () => $'IO.print (toString !x)' : ()
        C = \ () => write_c x
        Pascal = \ () => $'Write(!x)' : ()
        Kotlin = \ () =>
            $'run { System.out.write(!x.toString().toByteArray(Charsets.UTF_8)); System.out.flush() }' : ()
        Swift = \ () => $'print(!x, terminator: "")' : ()
        Wasm = \ () => write_wasm x
    }

#### write_and_write_line_print_without_failing (test)

In [3]:
///- --test --symbol
///> rust
///> zig

write "a"
write_line "b"

.rs (rust)
ab
.zig (zig)
ab

0

In [4]:
:= write_ln l : () =
    write l
    backend_switch {
        Gleam = \ () => write_line () : ()
        Lua = \ () => write_line () : ()
        Fsharp = \ () => write_line () : ()
        Python = \ () => $"print()" : ()
        Cuda = \ () => $'printf("\\n")' : ()
        Rust = \ () => $'println\!()' : ()
        Zig = \ () => $'spiralPrint(&[_]u8{10})' : ()
        Lean = \ () => $'IO.println ""' : ()
    }

:= color_reset () =
    run_target \|
        | Zig => \ () =>
            $'"\\u{001b}[0m"' : string
        | Gleam _ => \ () => $'"\\u{001b}[0m"'
        | Lua _ => \ () => $'""'
        | Rust => \ () => $'Rc::<str>::from("\\u{001b}[0m")' : string
        | _ => \ () => $'"\\u001b[0m"'

:= color_bright_black () =
    run_target \|
        | Zig => \ () =>
            $'"\\u{001b}[90m"' : string
        | Gleam _ => \ () => $'"\\u{001b}[90m"'
        | Lua _ => \ () => $'""'
        | Rust => \ () => $'Rc::<str>::from("\\u{001b}[90m")' : string
        | _ => \ () => $'"\\u001b[90m"'

:= color_bright_red () =
    run_target \|
        | Zig => \ () =>
            $'"\\u{001b}[91m"' : string
        | Gleam _ => \ () => $'"\\u{001b}[91m"'
        | Lua _ => \ () => $'""'
        | Rust => \ () => $'Rc::<str>::from("\\u{001b}[91m")' : string
        | _ => \ () => $'"\\u001b[91m"'

:= color_bright_green () =
    run_target \|
        | Zig => \ () =>
            $'"\\u{001b}[92m"' : string
        | Gleam _ => \ () => $'"\\u{001b}[92m"'
        | Lua _ => \ () => $'""'
        | Rust => \ () => $'Rc::<str>::from("\\u{001b}[92m")' : string
        | _ => \ () => $'"\\u001b[92m"'

:= color_yellow () =
    run_target \|
        | Zig => \ () =>
            $'"\\u{001b}[93m"' : string
        | Gleam _ => \ () => $'"\\u{001b}[93m"'
        | Lua _ => \ () => $'""'
        | Rust => \ () => $'Rc::<str>::from("\\u{001b}[93m")' : string
        | _ => \ () => $'"\\u001b[93m"'

:= color_bright_blue () =
    run_target \|
        | Zig => \ () =>
            $'"\\u{001b}[94m"' : string
        | Gleam _ => \ () => $'"\\u{001b}[94m"'
        | Lua _ => \ () => $'""'
        | Rust => \ () => $'Rc::<str>::from("\\u{001b}[94m")' : string
        | _ => \ () => $'"\\u001b[94m"'

:= color_bright_magenta () =
    run_target \|
        | Zig => \ () =>
            $'"\\u{001b}[95m"' : string
        | Gleam _ => \ () => $'"\\u{001b}[95m"'
        | Lua _ => \ () => $'""'
        | Rust => \ () => $'Rc::<str>::from("\\u{001b}[95m")' : string
        | _ => \ () => $'"\\u001b[95m"'

:= color_bright_cyan () =
    run_target \|
        | Zig => \ () =>
            $'"\\u{001b}[96m"' : string
        | Gleam _ => \ () => $'"\\u{001b}[96m"'
        | Lua _ => \ () => $'""'
        | Rust => \ () => $'Rc::<str>::from("\\u{001b}[96m")' : string
        | _ => \ () => $'"\\u001b[96m"'

:= color_bright_white () =
    run_target \|
        | Zig => \ () =>
            $'"\\u{001b}[97m"' : string
        | Gleam _ => \ () => $'"\\u{001b}[97m"'
        | Lua _ => \ () => $'""'
        | Rust => \ () => $'Rc::<str>::from("\\u{001b}[97m")' : string
        | _ => \ () => $'"\\u001b[97m"'